# 🧹 Processo de ETL (Extração, Transformação e Carga)

Este notebook executa o pipeline completo de processamento de dados do projeto de Estatística Aplicada. Ele é responsável por:
1. Carregar os arquivos JSON brutos localizados na pasta `bases/`.
2. Padronizar os nomes das secretarias/órgãos municipais.
3. Agrupar os dados financeiro (despesas), adiantamentos e quadro de cargos por mês e órgão.
4. Cruzar e expandir os dados dos Agentes Públicos (servidores ativos) gerando uma base consolidada no tempo.
5. Calcular as variáveis chave: `vol_adiantamentos_mensal`, `proporcao_comissionado_mensal` e a nossa variável alvo `target_instabilidade`.
6. Exportar o dataset unificado para `base_unificada.csv` contendo mais de 20 mil registros.

In [ ]:
import os
import json
import pandas as pd
import numpy as np

# Dicionário para padronização dos nomes dos órgãos/secretarias municipais
mapeamento_orgaos = {
    'SECRETARIA MUNICIPAL DA FAZENDA': 'SEC. MUN. DA FAZENDA',
    'SECRETARIA MUNICIPAL DE SAUDE': 'FUNDO MUNICIPAL DE SAÚDE',
    'SECRETARIA MUNICIPAL DE EDUCACAO': 'SEC. MUN. DE EDUCAÇÃO',
    'SECRETARIA MUNICIPAL DE ASSISTENCIA SOCIAL': 'FUNDO MUN. DE ASSIST. SOCIAL',
    'SECRETARIA MUNICIPAL INFRAESTRUTURA E MOB URBANA': 'SEC. M. DE INFRAEST.,PLANEJ. E MOBILIDADE URBANA',
    'SECRETARIA DE DESENV. ECONÔMICO, INDÚSTRIA, COMÉRCIO E SERVIÇOS': 'SEC. DES. ECONÔMICO, INDÚSTRIA, COMÉRCIO E SERVIÇO',
    'GABINETE DO PREFEITO': 'GABINETE DO PREFEITO',
    'FUNDO MUNICIPAL DE SAUDE DE CRICIUMA': 'FUNDO MUNICIPAL DE SAÚDE',
    'FUNDO MUNICIPAL DE ASSISTENCIA SOCIAL DE CRICIUMA': 'FUNDO MUN. DE ASSIST. SOCIAL',
    'PREFEITURA MUNICIPAL DE CRICIUMA': 'GABINETE DO PREFEITO'
}

# Ajuste robusto de caminhos para rodar a partir do diretório raiz ou da pasta 'src'
base_dir = "bases" if os.path.exists("bases") else "../bases"
export_path = "bases/base_unificada.csv" if os.path.exists("bases") else "../bases/base_unificada.csv"

print(f"Diretório de bases de dados configurado: {os.path.abspath(base_dir)}")
print(f"Caminho de exportação configurado: {os.path.abspath(export_path)}")

Diretório de bases de dados configurado: c:\Users\Felipe\Downloads\grafico_dispersao_XY\bases
Caminho de exportação configurado: c:\Users\Felipe\Downloads\grafico_dispersao_XY\base_unificada.csv


In [2]:
def load_and_concat(base_name):
    """Carrega e concatena todos os arquivos JSON que iniciam com o nome da base."""
    dfs = []
    if not os.path.exists(base_dir):
        print(f"Erro: Pasta {base_dir} não encontrada!")
        return pd.DataFrame()
        
    for f in os.listdir(base_dir):
        if f.startswith(base_name) and f.endswith(".json"):
            caminho_completo = os.path.join(base_dir, f)
            print(f"- Carregando: {f} ({os.path.getsize(caminho_completo) / (1024*1024):.2f} MB)")
            with open(caminho_completo, "r", encoding="utf-8") as file:
                data = json.load(file)
                if isinstance(data, list):
                    dfs.append(pd.json_normalize(data))
                elif isinstance(data, dict):
                    for k, v in data.items():
                        if isinstance(v, list):
                            dfs.append(pd.json_normalize(v))
                            break
                    else:
                        dfs.append(pd.json_normalize([data]))
                        
    if dfs:
        df_concatenado = pd.concat(dfs, ignore_index=True)
        print(f"  -> Total de registros carregados para '{base_name}': {len(df_concatenado)}")
        return df_concatenado
    else:
        print(f"  -> Nenhuma base encontrada para '{base_name}'")
        return pd.DataFrame()

def padronizar_orgao(df):
    """Padroniza o nome do órgão de acordo com o dicionário de mapeamento."""
    if df.empty: 
        return df
    col_found = next((c for c in ["orgao", "nomeEntidade", "descricaoOrgao", "credor"] if c in df.columns), None)
    if col_found:
        df["orgao_padronizado"] = df[col_found].str.upper().str.strip().replace(mapeamento_orgaos)
    else:
        df["orgao_padronizado"] = "DESCONHECIDO"
    return df

def convert_to_numeric(series):
    """Converte strings monetárias no padrão brasileiro (1.000,00) para numeric float."""
    return pd.to_numeric(series.astype(str).str.replace(',', '.'), errors='coerce')

## ⚙️ Execução do Pipeline ETL

Agora iremos processar e consolidar as 4 bases de dados principais. 

In [3]:
print("1. Carregando e Padronizando as Bases...")
df_ad = padronizar_orgao(load_and_concat("Adiantamentos"))
df_ag = padronizar_orgao(load_and_concat("Agentes Públicos"))
df_de = padronizar_orgao(load_and_concat("Despesas com Pessoal"))
df_qu = padronizar_orgao(load_and_concat("Quadro de Cargos"))

print("\n2. Agregando Variáveis Financeiras por Mês/Ano...")

# ADIANTAMENTOS
if not df_ad.empty and 'dataPagamento' in df_ad.columns:
    df_ad['mes_ano'] = pd.to_datetime(df_ad['dataPagamento'], errors='coerce').dt.to_period('M').astype(str)
    df_ad['valorPagamentoNum'] = convert_to_numeric(df_ad['valorPagamento']).fillna(0)
    ad_cols = [c for c in ['funcao', 'categoriaEconomica', 'tipo', 'numero', 'numeroDespesa', 'numeroEmpenho', 'fonteRecurso', 'programa', 'unidade', 'naturezaDespesa', 'elementoDespesa', 'subfuncao', 'acao'] if c in df_ad.columns]
    ad_agg = df_ad.groupby(['orgao_padronizado', 'mes_ano']).agg({
        'valorPagamentoNum': 'sum', **{c: 'first' for c in ad_cols}
    }).reset_index().rename(columns={'valorPagamentoNum': 'vol_adiantamentos_mensal'})
    ad_agg.rename(columns={c: f"ad_{c}" for c in ad_cols}, inplace=True)
    print(f"- Adiantamentos agrupados: {ad_agg.shape}")
else: 
    ad_agg = pd.DataFrame(columns=['orgao_padronizado', 'mes_ano', 'vol_adiantamentos_mensal'])

# DESPESAS COM PESSOAL
if not df_de.empty and 'dataEmpenho' in df_de.columns:
    df_de['mes_ano'] = pd.to_datetime(df_de['dataEmpenho'], errors='coerce').dt.to_period('M').astype(str)
    val_col = 'valorLiquidadoEmpenho' if 'valorLiquidadoEmpenho' in df_de.columns else 'valorEmpenhado'
    df_de['valor_despesa'] = convert_to_numeric(df_de[val_col]).fillna(0)
    
    de_cols = [c for c in ['idProjetoAtividade', 'idFuncao', 'idPrograma', 'descricaoPrograma', 'dotacaoOrcamentaria', 'categoriaEmpenho', 'idElemento', 'tipoEmpenho', 'tipoRecurso', 'descricaoElemento', 'idRecurso', 'descricaoFuncao', 'descricaoSubfuncao', 'descricaoUnidade', 'modalidadeAplicacao', 'saldoAPagar', 'valorRestosAPagarProcessados', 'ValorAnuladoEmpenho'] if c in df_de.columns]
    
    de_agg = df_de.groupby(['orgao_padronizado', 'mes_ano']).agg({
        'valor_despesa': 'sum', **{c: 'first' for c in de_cols}
    }).reset_index().rename(columns={'valor_despesa': 'gastos_pessoal_mensal'})
    de_agg.rename(columns={c: f"de_{c}" for c in de_cols}, inplace=True)
    print(f"- Despesas com pessoal agrupadas: {de_agg.shape}")
else: 
    de_agg = pd.DataFrame(columns=['orgao_padronizado', 'mes_ano', 'gastos_pessoal_mensal'])

# QUADRO DE CARGOS
if not df_qu.empty and 'competencia' in df_qu.columns:
    df_qu['mes_ano'] = pd.to_datetime(df_qu['competencia'], errors='coerce').dt.to_period('M').astype(str)
    df_qu['is_comissionado'] = df_qu.get('classificacaoCargo', pd.Series(dtype=str)).str.lower().str.contains('comissionado').fillna(False)
    vagas_col = 'quantidadeVagasCriadas'
    if vagas_col in df_qu.columns:
        df_qu['vagas_num'] = convert_to_numeric(df_qu[vagas_col]).fillna(1)
    else:
        df_qu['vagas_num'] = 1
    
    qu_cols = [c for c in ['nivelEscolaridade', 'situacaoCargo', 'classificacaoCargo', 'requisitos', 'leiCargo', 'cargo', 'atividades'] if c in df_qu.columns]
    
    qu_agg = df_qu.groupby(['orgao_padronizado', 'mes_ano']).agg({
        'vagas_num': 'sum',
        'is_comissionado': 'sum',
        **{c: 'first' for c in qu_cols}
    }).reset_index()
    qu_agg['proporcao_comissionado_mensal'] = qu_agg['is_comissionado'] / qu_agg['vagas_num'].replace(0, 1)
    qu_agg.rename(columns={c: f"qu_{c}" for c in qu_cols}, inplace=True)
    print(f"- Quadro de cargos agrupado: {qu_agg.shape}")
else: 
    qu_agg = pd.DataFrame(columns=['orgao_padronizado', 'mes_ano'])

print("\n3. Construindo Base Unificada e Expandindo Agentes no Tempo (Série de 2025)...")
meses_2025 = [f"2025-{str(i).zfill(2)}" for i in range(1, 13)]

ag_num_cols = ['valorRemuneracaoContratual', 'cargaHorariaSemanal', 'cargaHorariaMensal']
for c in ag_num_cols:
    if c in df_ag.columns: 
        df_ag[f'agnum_{c}'] = convert_to_numeric(df_ag[c]).fillna(0)

if 'dataAdmissao' in df_ag.columns: 
    df_ag['agnum_diasDeCasa'] = (pd.to_datetime('2026-01-01') - pd.to_datetime(df_ag['dataAdmissao'], errors='coerce')).dt.days.fillna(0)

records = df_ag.to_dict(orient='records')
expanded_rows = []
for r in records:
    for m in meses_2025:
        new_r = r.copy()
        new_r['mes_ano'] = m
        expanded_rows.append(new_r)
base = pd.DataFrame(expanded_rows)
print(f"- Base inicial de Agentes expandida para {len(meses_2025)} meses: {base.shape}")

# Cruzando tudo via merge esquerdo
base = pd.merge(base, ad_agg, on=['orgao_padronizado', 'mes_ano'], how='left')
base = pd.merge(base, de_agg, on=['orgao_padronizado', 'mes_ano'], how='left')
base = pd.merge(base, qu_agg, on=['orgao_padronizado', 'mes_ano'], how='left')

# Preenchendo nulos com 0 para valores fundamentais de gastos/adiantamentos
base['vol_adiantamentos_mensal'] = base.get('vol_adiantamentos_mensal', pd.Series([0])).fillna(0)
base['gastos_pessoal_mensal'] = base.get('gastos_pessoal_mensal', pd.Series([0])).fillna(0)

# DEFININDO A VARIÁVEL ALVO (Instabilidade = Gasto de Pessoal + Adiantamento Emergencial)
base['target_instabilidade'] = base['gastos_pessoal_mensal'] + base['vol_adiantamentos_mensal']

print(f"\n💾 Exportando base consolidada para: {os.path.abspath(export_path)}")
base.to_csv(export_path, index=False)
print(f"✅ Sucesso! Foram exportadas {base.shape[0]} linhas e {base.shape[1]} colunas.")

1. Carregando e Padronizando as Bases...
- Carregando: Adiantamentos-2025.json (0.42 MB)
- Carregando: Adiantamentos-2026.json (0.16 MB)
  -> Total de registros carregados para 'Adiantamentos': 437
- Carregando: Agentes Públicos-Trabalhando.json (15.33 MB)
  -> Total de registros carregados para 'Agentes Públicos': 5875
- Carregando: Despesas com Pessoal-2025.json (7.67 MB)
- Carregando: Despesas com Pessoal-2026.json (2.50 MB)
  -> Total de registros carregados para 'Despesas com Pessoal': 3758
- Carregando: Quadro de Cargos-2025.json (35.42 MB)
- Carregando: Quadro de Cargos-2026.json (15.02 MB)
  -> Total de registros carregados para 'Quadro de Cargos': 13027

2. Agregando Variáveis Financeiras por Mês/Ano...
- Adiantamentos agrupados: (85, 16)
- Despesas com pessoal agrupadas: (48, 20)
- Quadro de cargos agrupado: (17, 12)

3. Construindo Base Unificada e Expandindo Agentes no Tempo (Série de 2025)...
- Base inicial de Agentes expandida para 12 meses: (70500, 32)

💾 Exportando base

In [4]:
# Visualização rápida dos dados gerados
print(f"Dimensões da base unificada: {base.shape}")
base[['orgao_padronizado', 'mes_ano', 'vol_adiantamentos_mensal', 'gastos_pessoal_mensal', 'target_instabilidade']].head(10)

Dimensões da base unificada: (70500, 75)


,orgao_padronizado,mes_ano,vol_adiantamentos_mensal,gastos_pessoal_mensal,target_instabilidade
0,GABINETE DO PREFEITO,2025-01,0.0,24933200.82,24933200.82
1,GABINETE DO PREFEITO,2025-02,0.0,16814000.19,16814000.19
2,GABINETE DO PREFEITO,2025-03,0.0,18106629.58,18106629.58
3,GABINETE DO PREFEITO,2025-04,15000.0,17628502.20,17643502.20
4,GABINETE DO PREFEITO,2025-05,3734.0,21094917.03,21098651.03
5,GABINETE DO PREFEITO,2025-06,1734.4,23340167.07,23341901.47
6,GABINETE DO PREFEITO,2025-07,0.0,29727713.22,29727713.22
7,GABINETE DO PREFEITO,2025-08,5200.0,11521037.53,11526237.53
8,GABINETE DO PREFEITO,2025-09,2500.0,21492833.66,21495333.66
9,GABINETE DO PREFEITO,2025-10,0.0,20994093.32,20994093.32
